# Step 0 — Setup and data acquisition

**Project:** Network IDS with GAN-based data augmentation (CSE-CIC-IDS2018)

| | |
|---|---|
| **Goal** | Get the official dataset into a reliable, fast format and take a first inventory of what we have. |
| **Inputs** | 10 daily CSV files from the public CSE-CIC-IDS2018 S3 bucket (~6.9 GB) |
| **Outputs** | `data/interim/<day>.parquet` (lossless copy) + `results/tables/inventory_*.csv` + a class-distribution figure |
| **Tools** | Google Colab (CPU), Google Drive, pandas, PyArrow |

### Principle of this step: *acquire, don't decide*
We **do not clean** the data here. Removing NaN/Inf values, duplicates or rare classes changes
the dataset, and those are **methodological decisions**. We take them in the preprocessing step,
after the EDA, and justify them in the report. This step only:
1. downloads the files from the **official source** (traceability);
2. converts them to **Parquet**: typed, compressed, column-oriented, 10–50× faster to read than CSV;
3. removes what is clearly **not data** (repeated header lines inside the files) or must **not be used** (IP addresses);
4. produces an **inventory** (rows, labels and time span per day).

## 1. Environment

**Why Colab?** We have no local GPU and the dataset is larger than a typical laptop's RAM.
Colab gives a free VM (~12 GB RAM, ~100 GB temporary disk), and a GPU if we ever need one.
For tabular data, CPU is enough: our GAN and MLP will be small fully-connected networks.

**Storage strategy**
- **Code:** GitHub repo, cloned into Google Drive, so it survives Colab restarts.
- **Raw CSVs (6.9 GB):** Colab's *temporary* local disk `/content/raw`. They are only needed once, and the free Drive (15 GB) is better kept for the processed data.
- **Parquet + results:** inside the project folder on Drive (persistent). `data/` is git-ignored, because the course forbids committing datasets.

➡️ Run the cell. Colab will ask for permission to access your Drive.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/dija040/ids-project.git"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/ids-project")

    def git(*args):
        # Run a git command; if it fails, show git's own error message.
        r = subprocess.run(["git", *args], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("git " + " ".join(args) + " failed: " + r.stderr)
        return r.stdout.strip()

    if not (PROJECT_DIR / ".git").exists():
        git("clone", REPO_URL, str(PROJECT_DIR))
    else:
        # The Drive copy is only used to RUN the code (we edit it on the PC and push to
        # GitHub), so we force it to match GitHub exactly instead of merging with `pull`.
        # Untracked files (data/, generated results) are not touched.
        git("-C", str(PROJECT_DIR), "fetch", "origin")
        git("-C", str(PROJECT_DIR), "reset", "--hard", "origin/main")
    print("Code version:", git("-C", str(PROJECT_DIR), "log", "-1", "--format=%h %s"))
    os.environ["IDS_RAW_DIR"] = "/content/raw"      # raw CSVs on the fast temporary disk
else:
    PROJECT_DIR = Path.cwd().resolve().parent          # running locally from notebooks/

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Project directory:", PROJECT_DIR)

In [ ]:
# Colab already ships most libraries; this only installs what is missing.
%pip install -q -r requirements.txt

In [ ]:
import platform, json
import numpy as np, pandas as pd, pyarrow as pa
import matplotlib.pyplot as plt

from src import config, data_io
config.set_seed()          # reproducibility: same seed everywhere
config.ensure_dirs()

print("Python ", platform.python_version())
print("pandas ", pd.__version__, "| numpy", np.__version__, "| pyarrow", pa.__version__)
print("RAW_DIR     =", config.RAW_DIR)
print("INTERIM_DIR =", config.INTERIM_DIR)

## 2. The dataset: CSE-CIC-IDS2018

**What it is.** A dataset built in 2018 by the Communications Security Establishment (CSE) and the
Canadian Institute for Cybersecurity (CIC). They deployed a realistic company network on AWS
(about 420 victim machines and 30 servers in 5 departments, plus an attacking infrastructure),
generated benign user behaviour with *profiles*, and ran 7 attack scenarios over 10 working days.

**What a sample is.** The raw packet captures (PCAP) were turned into **bidirectional flows** with the
tool **CICFlowMeter**. A flow is all the packets exchanged between two endpoints (same IPs, ports and
protocol) until a FIN packet or a timeout. Each flow is described by **~80 statistics**: duration,
packet counts and sizes, inter-arrival times (IAT), TCP flags, window sizes, active/idle times, etc.
An IDS built on this data therefore judges **behaviour**, not content: it never sees payloads.

**Why the "Processed Traffic Data for ML Algorithms" folder?** These are the CSVs already produced by
CICFlowMeter, with one label per flow. The PCAPs (~450 GB) would only be needed to re-extract features.

| Day | Attacks (plus benign traffic every day) |
|---|---|
| Wed 14/02 | FTP-BruteForce, SSH-Bruteforce |
| Thu 15/02 | DoS GoldenEye, DoS Slowloris |
| Fri 16/02 | DoS SlowHTTPTest, DoS Hulk |
| Tue 20/02 | DDoS LOIC-HTTP, DDoS LOIC-UDP |
| Wed 21/02 | DDoS LOIC-UDP, DDoS HOIC |
| Thu 22/02 | Brute Force-Web, Brute Force-XSS, SQL Injection |
| Fri 23/02 | Brute Force-Web, Brute Force-XSS, SQL Injection |
| Wed 28/02 | Infiltration |
| Thu 01/03 | Infiltration |
| Fri 02/03 | Bot |

⚠️ Notice already: **each attack exists only on 1–2 days**. This will drive our split strategy (step 2).

*Citation (mandatory):* I. Sharafaldin, A. H. Lashkari, A. A. Ghorbani, "Toward Generating a New Intrusion
Detection Dataset and Intrusion Traffic Characterization", ICISSP 2018.

In [ ]:
# The bucket is public: we can list it over plain HTTPS, no AWS account needed.
remote = data_io.list_remote_files()
print(f"{len(remote)} files, total {remote['size_bytes'].sum()/1e9:.2f} GB")
remote

## 3. Download

The files are streamed straight from the official bucket. Each file is first written as `.part` and
renamed only when its size matches the one announced by S3. This way, an interrupted download is
never mistaken for a complete one. Re-running the cell skips files that are already complete.

⏱️ Expect roughly 5–15 minutes on Colab.

In [ ]:
sizes = dict(zip(remote["file"], remote["size_bytes"]))
for day, (fname, _) in config.DAY_FILES.items():
    data_io.download_file(fname, config.RAW_DIR, expected_size=sizes[fname])
for f in sorted(config.RAW_DIR.glob("*.csv")):
    print(f"{f.stat().st_size/1e6:>9,.0f} MB  {f.name}")

## 4. Conversion CSV → Parquet

**What `csv_to_parquet` does** (see `src/data_io.py`, it is commented line by line):

| Operation | Why |
|---|---|
| Read in chunks of 500,000 rows | The 20/02 file is 4 GB (~8 M flows) and would not fit in RAM at once. |
| Read everything as text first | Some files contain **repeated header lines** and values such as `Infinity`. Automatic type guessing would silently turn whole columns into text. |
| Remove repeated header lines | They are artefacts of how the CSVs were concatenated, not network flows. We **count** them for the report. |
| Drop `Flow ID`, `Src IP`, `Src Port`, `Dst IP` (20/02 file only) | (1) same schema for all 10 files; (2) privacy: the course forbids publishing IPs; (3) **shortcut learning**: in a testbed the attacker IPs are fixed, so a model would learn "this IP = attack". That rule is useless on any other network. |
| Features → `float64`, `Timestamp` → datetime | Correct types for analysis. We use float64 rather than float32 because some counters reach 10⁸ and float32 would round them. |
| `Infinity` / `NaN` kept as `inf` / `NaN` | They will be **counted and handled** in preprocessing, not hidden now. |
| Add `day` and `row_in_file` | Needed for the **time-aware split**. The original order is also a tie-breaker, because timestamps are only precise to the second. |

⏱️ Roughly 15–25 minutes in total (the 20/02 file takes most of it). Days already converted are skipped.

In [ ]:
import time

summaries = []
for day, (fname, _) in config.DAY_FILES.items():
    out = config.INTERIM_DIR / f"{day}.parquet"
    summary_file = config.INTERIM_DIR / f"{day}.summary.json"
    if out.exists() and summary_file.exists():
        print(f"[skip] {day} already converted")
        summaries.append(json.loads(summary_file.read_text()))
        continue
    t0 = time.time()
    s = data_io.csv_to_parquet(config.RAW_DIR / fname, out, day=day)
    summary_file.write_text(json.dumps(s, indent=2))
    summaries.append(s)
    print(f"[done] {day}: {s['rows']:>10,} flows | {time.time()-t0:6.1f}s | "
          f"header lines removed: {s['header_rows_removed']} | "
          f"bad timestamps: {s['unparsed_timestamps']} | "
          f"Parquet {out.stat().st_size/1e6:,.0f} MB")

## 5. First inventory

Before any EDA, we want a few basic facts: **how much data, per day, per label, over what time span**.
Those numbers go straight into the *Dataset description* section of the report.

In [ ]:
inv = pd.DataFrame([{k: v for k, v in s.items() if k != "label_counts"} for s in summaries])
inv["dropped_identifier_columns"] = inv["dropped_identifier_columns"].apply(lambda c: ", ".join(c) or "-")
inv.to_csv(config.TABLES_DIR / "inventory_per_day.csv", index=False)
print(f"TOTAL: {inv['rows'].sum():,} flows")
inv

In [ ]:
# Label x day matrix: shows which attack lives on which day.
label_day = (pd.DataFrame({s["day"]: s["label_counts"] for s in summaries})
               .fillna(0).astype(int))
label_day["TOTAL"] = label_day.sum(axis=1)
label_day = label_day.sort_values("TOTAL", ascending=False)
label_day.to_csv(config.TABLES_DIR / "inventory_label_by_day.csv")
label_day

In [ ]:
dist = label_day["TOTAL"].rename("flows").to_frame()
dist["percent"] = (100 * dist["flows"] / dist["flows"].sum()).round(4)
dist["imbalance_ratio_vs_benign"] = (dist.loc["Benign", "flows"] / dist["flows"]).round(1)
print(dist)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(dist.index[::-1], dist["flows"][::-1], color="#4C72B0")
ax.set_xscale("log")    # log scale: otherwise only the benign bar would be visible
ax.set_xlabel("Number of flows (log scale)")
ax.set_title("CSE-CIC-IDS2018: class distribution (all 10 days)")
for i, v in enumerate(dist["flows"][::-1]):
    ax.text(v * 1.1, i, f"{v:,}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "00_class_distribution.png", dpi=150)
plt.show()

In [ ]:
# Sanity check: re-read one Parquet file and verify types and contents.
check = pd.read_parquet(config.INTERIM_DIR / "2018-03-01.parquet")
print(check.shape)
print(check.dtypes.value_counts())
check.head()

## 6. What to look at in the results (bring these to our next discussion)

1. **Total number of flows** and the share of `Benign`. How imbalanced is the dataset?
2. **The smallest classes.** Which attacks have fewer than ~1,000 flows? These are the GAN's targets.
3. **Imbalance ratio.** How many benign flows exist for each flow of the rarest attack?
4. **Header lines removed / bad timestamps.** Which files had artefacts? (This is data quality, for the report.)
5. **The label × day matrix.** Confirm that each attack is confined to its documented days.
6. Any label you **did not expect** (e.g. a strange spelling, or a label appearing on the wrong day).

**Next step (01, EDA):** missing and infinite values, duplicates, constant features, feature
distributions per class, correlations, and a look at the known problems of this dataset.

> 📝 After running, fill in `reports/step_reports/00_setup_and_acquisition.md` with the numbers you obtained.